In [ ]:
import os
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

# figure 1

# setup paths and load data
drive_data_dir = "/content/drive/MyDrive/data"
parquet_path = os.path.join(drive_data_dir, "cleaned_patent_panel.parquet")

con = duckdb.connect()
df_sec = con.execute(f"""
    SELECT
        cpc_section,
        CASE WHEN vc_backed = 1 THEN 'VC-Backed' ELSE 'Non-VC' END AS vc_status,
        normalized_forward_citations
    FROM '{parquet_path}'
    WHERE cpc_section IS NOT NULL
      AND cpc_section NOT IN ('UNKNOWN', 'D', 'E')
      AND normalized_forward_citations IS NOT NULL;
""").df()

plt.rcParams.update({
    'font.size': 10,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': '#000000',
    'grid.color': '#000000',
    'grid.alpha': 0.25
})

fig, ax = plt.subplots(figsize=(8.5, 5), dpi=300)

sns.barplot(
    data=df_sec,
    x='cpc_section',
    y='normalized_forward_citations',
    hue='vc_status',
    palette={'VC-Backed': '#4da6ff', 'Non-VC': '#005a32'},
    edgecolor='#000000',
    linewidth=0.8,
    errorbar='se',
    capsize=0.1,
    err_kws={'linewidth': 0.8, 'color': '#000000'},
    ax=ax
)

ax.axhline(1.0, color='#000000', linestyle='--', linewidth=1.0, alpha=0.85, label='Sector Benchmark (=1.0)')

ax.set_title('Figure 1: Mean Normalized Forward Citation Impact Across CPC Technology Sections', fontsize=11, fontweight='bold', pad=14)
ax.set_xlabel('CPC Technology Section', fontsize=10, labelpad=8)
ax.set_ylabel('Mean Normalized Forward Citations', fontsize=10, labelpad=8)
ax.grid(True, linestyle=':', axis='y')

ax.legend(title='Funding Status', frameon=True, loc='upper left')

caption = (
    "Figure 1: Mean normalized forward citations by Cooperative Patent Classification (CPC) section and funding status (2000–2020).\n"
    "Normalized citations are computed relative to the peer benchmark (CPC subclass × grant year mean = 1.0, dashed line).\n"
    "Sections D (Textiles) and E (Fixed Constructions) are excluded due to insufficient observation counts in the VC cohort.\n"
    "Included CPC Sections: A = Human Necessities, B = Operations & Transport, C = Chemistry & Metallurgy, "
    "F = Mechanical Engineering, G = Physics & Computing, H = Electricity."
)
plt.figtext(0.5, -0.15, caption, wrap=True, horizontalalignment='center', fontsize=8.5, color='#000000')

plt.tight_layout()

fig1_path = os.path.join(drive_data_dir, "figure1_cpc_section_citations.png")
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.show()